# 4-1. RAG 기반 AI 에이전트 — 이론과 실습

---

## 목차

| # | 내용 |
|:---:|------|
| 0 | **환경 설정** — 패키지 설치, API 키, 모델 초기화 |
| 1 | **LLM의 한계** — 환각(Hallucination) 현상 직접 체험 |
| 2 | **RAG란 무엇인가** — 개념, 아키텍처, 구성요소, 실무 사례 |
| 3 | **검색의 진화** — 키워드 검색 → 의미 기반 검색 |
| 4 | **임베딩과 Vector Store** — 텍스트를 숫자로, 숫자를 DB로 |
| 5 | **청킹 전략** — 문서를 어떻게 잘게 나눌 것인가 |
| 6 | **LangGraph 기초 + RAG 파이프라인** — 구성요소, 기본 문법, 조건부 엣지, RAG 적용 |

---

## 0. 환경 설정


In [ ]:
!pip install -q \
    langchain>=1.0.0 \
    langchain-openai>=1.0.0 \
    langchain-upstage>=0.3.0 \
    langchain-community>=0.3.0 \
    langgraph>=1.0.0 \
    langchain-text-splitters>=1.0.0 \
    chromadb>=0.5.0 \
    tiktoken>=0.7.0 \
    pymupdf>=1.24.0 \
    python-dotenv>=1.0.0

In [5]:
import warnings
import os
from os import getenv
from dotenv import load_dotenv
warnings.filterwarnings('ignore')

load_dotenv()

# API 키 확인 (.env 미설정 시 직접 입력)
# OPENAI_API_KEY = getenv('OPENAI_API_KEY')
# if OPENAI_API_KEY:
#     print('API 키 로드 성공!')
# else:
#     print('ERROR: .env 파일에 OPENAI_API_KEY가 설정되지 않았습니다.')

# Upstage API 사용 시 필요 (Option B 선택 시)
UPSTAGE_API_KEY = getenv('UPSTAGE_API_KEY')
if UPSTAGE_API_KEY:
    print('API 키 로드 성공!')
else:
    print('ERROR: .env 파일에 UPSTAGE_API_KEY가 설정되지 않았습니다.')

print('환경 설정 완료')


ModuleNotFoundError: No module named 'dotenv'

In [ ]:
from langchain_openai import ChatOpenAI, OpenAIEmbeddings

# ═══════════════════════════════════════════════════════════
# 모델 초기화 — OpenAI / Solar 중 택 1 (주석 해제하여 전환)
# ═══════════════════════════════════════════════════════════

# ── Option A: OpenAI (기본) ──
# llm = ChatOpenAI(model='gpt-5-nano', temperature=0)
# embeddings = OpenAIEmbeddings(model='text-embedding-3-small')

# ── Option B: Solar 대안 (주석 해제하여 사용) ──
from langchain_upstage import ChatUpstage, UpstageEmbeddings
llm = ChatUpstage(model='solar-pro3')
embeddings = UpstageEmbeddings(model='embedding-query')

print(f'모델 초기화 완료')


In [ ]:
import glob

# 데이터 파일 경로 확인
# PDF 파일들이 data/ 폴더에 있어야 한다.
DATA_DIR = 'data/'
pdf_files = sorted(glob.glob(DATA_DIR + '*.pdf'))

print(f'PDF 파일 수: {len(pdf_files)}개')
for f in pdf_files:
    print(f'  {os.path.basename(f)}')

if not pdf_files:
    print('\n⚠️ data/ 폴더에 PDF 파일이 없습니다.')
    print('  실습 폴더 내 data/ 디렉토리에 Yes24 PDF 파일을 넣어주세요.')


---

## 1. LLM의 한계 — 왜 LLM만으로는 부족한가?

![Image F](https://i.ibb.co/Fbk5qXyS/image-f.png)

### 1-1. LLM의 세 가지 근본적 한계

LLM(Large Language Model)은 방대한 텍스트 데이터로 학습되어 다양한 질문에 답변할 수 있다.
하지만 다음과 같은 **근본적인 한계**가 있다.

| 한계 | 설명 | 예시 |
|------|------|------|
| **환각 (Hallucination)** | 학습 데이터에 없는 내용을 **그럴듯하게 지어냄** | 존재하지 않는 배송 정책을 자신 있게 설명 |
| **최신 정보 부재** | 학습 이후의 정보를 알 수 없음 | 2024년 변경된 정책 미반영 |
| **도메인 지식 부족** | 특정 회사/서비스의 내부 정보는 학습되지 않음 | Yes24 총알배송의 구체적 조건 |

<br>

> **💡 환각(Hallucination)이 특히 위험한 이유**
>
> LLM은 "모르겠습니다"라고 하지 않고, **자신감 있게 틀린 답변**을 한다.
> 고객 서비스에서 이런 답변이 나가면 고객 신뢰를 잃고, 법적 문제까지 발생할 수 있다.

아래에서 LLM에게 Yes24 서비스 정책을 직접 물어보고 환각이 발생하는지 확인해 보자.


In [ ]:
from langchain_core.messages import HumanMessage

# ========== LLM에게 직접 질문하기 ==========
# Yes24의 "총알배송"과 "배송지연 보상제도"는 회사 내부 정책이므로
# LLM의 학습 데이터에 정확히 포함되어 있지 않을 가능성이 높다.
questions = [
    'Yes24에서 총알배송이 뭔가요? 서울에서 당일배송 주문 마감 시간은?',
    'Yes24 배송지연 보상제도는 어떻게 되나요? 보상 금액은?',
]

for q in questions:
    response = llm.invoke([HumanMessage(content=q)])
    print(f'질문: {q}')
    print(f'응답: {response.content[:300]}...')
    print()

print('⚠️ 위 답변이 실제 Yes24 정책과 일치하는지 확인할 방법이 없다.')
print('   → 서울 당일배송 마감은 실제로 0~13시 (PDF 참조)')
print('   → 배송지연 보상은 주문 건당 YES포인트 2,000원 (PDF 참조)')


---

## 2. RAG란 무엇인가?

### 2-1. RAG의 정의

**RAG (Retrieval-Augmented Generation)** = 검색 증강 생성

LLM의 환각 문제를 해결하는 가장 효과적인 방법 중 하나이다.
핵심 아이디어는 단순하다: **"답변하기 전에, 먼저 관련 자료를 찾아서 읽고 답변하라."**

```
┌─ 기존 방식 (LLM만) ───────────────────────────────────┐
│  질문 ───→ LLM ───→ 답변 (환각 위험!)                 │
└───────────────────────────────────────────────────────┘

┌─ RAG 방식 ────────────────────────────────────────────┐
│  질문 ───→ [검색] ───→ 관련 문서 ───→ LLM ───→ 답변   │
│                    "이 자료를 참고해서"               │
└───────────────────────────────────────────────────────┘
```

> **💡 비유: 오픈북 시험**
>
> - **LLM만** = 책 없이 시험 보기 → 기억이 부정확하면 **그럴듯하게 지어냄**
> - **RAG** = 오픈북 시험 → 답을 모르면 **책에서 찾아서** 답변 → 정확도 대폭 향상

<br>

### 2-2. RAG 파이프라인의 전체 구조

RAG는 크게 **준비 단계(Indexing)**와 **실행 단계(Querying)** 두 가지로 나뉜다.

![이미지_A](https://i.ibb.co/jvXKKVt8/image-a.png)


```
┌──────────────── 준비 단계 (Indexing, 1회) ───────────────────┐
│                                                              │
│  [문서 수집]  →  [청킹]  →  [임베딩]  →  [Vector Store 저장] │
│   PDF 로드      잘게 자름    벡터 변환     DB에 저장         │
│                                                              │
└──────────────────────────────────────────────────────────────┘

┌──────────────── 실행 단계 (Querying, 매 질문마다) ─────────────┐
│                                                                │
│  [사용자 질문]  →  [검색]  →  [프롬프트 구성]  →  [LLM 생성]   │
│   "총알배송?"    유사 문서      자료+질문 결합    정확한 답변  │
│                                                                │
└────────────────────────────────────────────────────────────────┘
```

### 2-3. RAG의 핵심 구성요소

| 구성요소 | 역할 | 이 실습에서 사용하는 도구 |
|---------|------|:---:|
| **Document Loader** | 문서 파일(PDF, HTML 등)을 텍스트로 변환 | `PyMuPDFLoader` |
| **Text Splitter** | 긴 문서를 적절한 크기의 조각(chunk)으로 분할 | `RecursiveCharacterTextSplitter` |
| **Embedding Model** | 텍스트를 숫자 벡터로 변환 | `text-embedding-3-small` |
| **Vector Store** | 벡터를 저장하고 유사도 검색 수행 | `ChromaDB` |
| **Retriever** | 질문과 유사한 문서를 Vector Store에서 검색 | `vectorstore.as_retriever()` |
| **LLM** | 검색된 문서를 참고하여 최종 답변 생성 | `gpt-5-nano` |

### 2-4. RAG가 사용되는 실무 사례

| 분야 | 활용 사례 | 왜 RAG가 필요한가 |
|------|---------|------------------|
| **고객 서비스** | 사내 정책 기반 챗봇 (이 실습) | 사내 문서는 LLM 학습 데이터에 없음 |
| **법률** | 판례/법령 검색 기반 법률 자문 | 최신 법률 개정 사항 반영 필요 |
| **의료** | 의학 논문 기반 진단 보조 | 환각으로 인한 오진 방지 |
| **사내 검색** | 사내 위키/문서 기반 Q&A | 기업 내부 정보는 외부 LLM이 알 수 없음 |
| **교육** | 교재 기반 학습 도우미 | 특정 교재의 내용을 정확히 참조해야 함 |


### 2-5. RAG의 효과 — 자료를 주면 달라진다

실제로 관련 자료를 프롬프트에 포함시키면 답변이 어떻게 달라지는지 확인해 보자.


In [ ]:
from langchain_community.document_loaders import PyMuPDFLoader
from langchain_core.prompts import ChatPromptTemplate

# ========== PDF 문서 로드 ==========
# Yes24 총알배송 정책이 담긴 실제 문서를 읽어온다.
# 이 문서가 LLM에게 "오픈북"의 역할을 하게 된다.

# 총알배송 관련 PDF 파일 찾기
bullet_pdf = [f for f in pdf_files if '총알배송' in f]
if bullet_pdf:
    loader = PyMuPDFLoader(bullet_pdf[0])
    documents = loader.load()
    doc_content = '\n'.join([doc.page_content for doc in documents])
    print(f'문서 로드 완료: {os.path.basename(bullet_pdf[0])}')
    print(f'문서 길이: {len(doc_content)}자')
    print(f'\n내용 미리보기:\n{doc_content[:400]}...')
else:
    print('⚠️ 총알배송 PDF를 찾을 수 없습니다.')


In [ ]:
# ========== 자료를 포함하여 질문하기 (RAG의 핵심) ==========
# 시스템 프롬프트에 "참고 자료"를 포함시킨다.
# LLM은 이 자료를 기반으로 답변하므로 환각이 대폭 감소한다.
prompt_template = ChatPromptTemplate.from_messages([
    ('system', '''당신은 온라인 서점 Yes24의 고객 서비스 상담원입니다.
다음 자료를 참고하여 고객의 질문에 정확하게 답변해주세요.
자료에 없는 내용은 "해당 정보는 제공된 자료에 없습니다"라고 답변하세요.

[참고 자료]
{context}'''),
    ('human', '{question}')
])

question = 'Yes24에서 총알배송이 뭔가요? 서울에서 당일배송 주문 마감 시간은?'
messages = prompt_template.format_messages(context=doc_content, question=question)
response = llm.invoke(messages)

print(f'질문: {question}')
print(f'\n자료 기반 응답:')
print(response.content)
print('\n✅ 실제 PDF 문서에 기반한 정확한 답변!')


### 2-6. 그런데... 자료는 어떻게 "자동으로" 찾는가?

방금 실습에서는 **정답 문서를 우리가 직접 골라서** 프롬프트에 넣었다.
하지만 실제 서비스에서는 **수백~수천 개의 문서** 중에서 관련 문서를 **자동으로 찾아야** 한다.

```
사용자: "배송이 늦으면 보상받을 수 있나요?"

    → 총알배송 PDF? 배송지연 보상 PDF? 도서 품절 보상 PDF? 신규 회원 PDF?
      8개의 PDF 중 어떤 것이 관련 있는지 자동으로 판단해야 한다!
```

이 **"자동 검색"** 문제를 해결하는 것이 RAG 파이프라인의 핵심이다.
다음 챕터부터 검색 방법을 단계적으로 배워 보자.


---

## 3. 검색의 진화 — 키워드에서 의미로

### 3-1. 키워드 검색 (Lexical Search)

가장 단순한 검색 방법: **문자열이 정확히 포함되어 있는지** 확인한다.

```python
if '총알배송' in document:
    return document  # 문자열이 포함되면 반환
```

| 장점 | 단점 |
|------|------|
| 단순하고 빠르다 | **정확히 일치하는 키워드만** 찾을 수 있다 |
| 구현이 쉽다 | 동의어, 유사 표현을 전혀 인식하지 못한다 |

<br>

> **💡 핵심 문제**
>
> 고객이 "빠른 배송"이라고 질문하면 "총알배송" 문서를 **찾지 못한다.**
> 고객이 "포인트 유효기간"이라고 질문하면 "영원한 YES포인트" 문서를 **찾지 못한다.**
> 사람은 둘이 같은 의미인 걸 바로 알지만, 키워드 검색은 **글자가 다르면 다른 것**으로 취급한다.


In [ ]:
# ========== 모든 PDF 문서 로드 ==========
all_documents = []
for pdf_path in pdf_files:
    loader = PyMuPDFLoader(pdf_path)
    docs = loader.load()
    all_documents.extend(docs)
print(f'로드된 문서 수: {len(all_documents)}개 (PDF {len(pdf_files)}개)')

# ========== 키워드 검색 함수 ==========
def keyword_search(documents, keyword):
    """문서 리스트에서 키워드를 포함한 문서를 검색한다."""
    return [doc for doc in documents if keyword in doc.page_content]

# ========== 키워드 검색의 한계 테스트 ==========
# 같은 의미이지만 다른 표현으로 검색해 본다.
test_keywords = [
    ('총알배송', '정확한 키워드'),
    ('빠른 배송', '동의어'),
    ('배송 빨리', '유사 표현'),
    ('포인트 유효기간', '관련 표현 → "영원한 YES포인트"를 찾을 수 있을까?'),
]

print('\n키워드 검색 결과:')
for kw, desc in test_keywords:
    results = keyword_search(all_documents, kw)
    status = '✅' if results else '❌'
    print(f'  {status} "{kw}" ({desc}) → {len(results)}건')

print('\n→ 정확한 키워드만 찾고, 동의어/유사 표현은 전혀 인식하지 못한다.')


### 3-2. 의미 기반 검색 (Semantic Search)

키워드 검색의 한계를 극복하는 방법: **텍스트의 "의미"를 이해하는 검색.**

![image C](https://i.ibb.co/fdkcrxXD/image-c.png)

핵심 원리:
1. 모든 문서를 **숫자 벡터(임베딩)**로 변환하여 저장
2. 사용자의 질문도 **같은 방식으로 벡터로 변환**
3. 질문 벡터와 **가장 가까운(유사한) 문서 벡터**를 찾아 반환

```
"빠른 배송"   → [0.2, -0.5, 0.8, ...]  ← 질문 벡터
"총알배송"    → [0.3, -0.4, 0.7, ...]  ← 문서 벡터 (의미가 비슷 → 벡터도 비슷!)
"포인트 적립" → [0.9, 0.1, -0.3, ...]  ← 관련 없는 문서 (벡터가 멀다)
```

> **💡 2-1 챕터 연결**
>
> 2-1에서 "영화"와 유사한 단어를 벡터로 찾았던 것과 동일한 원리이다.
> 차이점: 이제 **단어가 아닌 문서 전체**를 벡터로 변환한다.


---

## 4. 임베딩과 Vector Store

### 4-1. 임베딩(Embedding) 복습

2-1 챕터에서 배운 핵심:
- 텍스트를 **고차원 벡터**(예: 1536차원)로 변환하는 기술
- 의미가 비슷한 텍스트는 **벡터 공간에서 가까운 위치**에 놓인다

| 구분 | 2-1 챕터 | 이번 챕터 |
|------|:---:|:---:|
| 대상 | 단어 단위 | 문장/문서 단위 |
| 모델 | `nn.Embedding` (768차원) | `text-embedding-3-small` (1536차원) |
| 용도 | 유사 단어 찾기 | **유사 문서 찾기 (RAG 검색)** |

![image D](https://i.ibb.co/1YnhvcxX/image-d.png)

### 4-2. 코사인 유사도 실습

실제 임베딩 벡터를 생성하고, Yes24 문서 관련 문장들의 유사도를 계산해 보자.


In [ ]:
import numpy as np

# ========== 임베딩 벡터 생성 + 유사도 계산 ==========
texts = [
    '빠른 배송 서비스가 궁금합니다',           # 질문
    '총알배송 표시상품을 주문하시면 당일에 받으실 수 있습니다',  # 관련 문서 (총알배송)
    'YES포인트는 유효기간이 없습니다',          # 관련 없는 문서 (포인트)
    '배송이 지연되면 보상을 받을 수 있나요',     # 다른 주제 (배송지연)
]

vectors = embeddings.embed_documents(texts)
print(f'벡터 차원: {len(vectors[0])}차원')

# 코사인 유사도 계산
def cosine_similarity(a, b):
    a, b = np.array(a), np.array(b)
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))

query_vec = vectors[0]  # "빠른 배송 서비스가 궁금합니다"
print(f'\n기준: "{texts[0]}"')
for i in range(1, len(texts)):
    sim = cosine_similarity(query_vec, vectors[i])
    print(f'  vs "{texts[i][:25]}..."  유사도: {sim:.4f}')

print('\n→ "총알배송"이 "빠른 배송"과 가장 유사하게 나온다!')
print('   키워드 검색에서는 불가능했던 것이 의미 기반 검색에서는 가능하다.')


### 4-3. Vector Store란?

모든 문서의 임베딩 벡터를 **저장하고 빠르게 검색**할 수 있는 특수 데이터베이스이다.

```
[Vector Store 동작 흐름]

① 저장 (인덱싱)  —  1회만 실행
   문서들 → 임베딩 모델 → 벡터들 → Vector Store에 저장

② 검색  —  매 질문마다 실행
   질문 → 임베딩 모델 → 질문 벡터 → Vector Store에서 유사한 벡터 검색 → 관련 문서 반환
```

대표적인 Vector Store:

| Vector Store | 특징 | 적합한 상황 |
|:---:|------|------|
| **ChromaDB** | 로컬 환경에서 간편하게 사용 | 실습, 프로토타이핑 (이 실습에서 사용) |
| **FAISS** | Meta에서 개발한 고속 검색 | 대규모 데이터, 속도 중시 |
| **Pinecone** | 클라우드 관리형 서비스 | 운영 환경, 확장성 중시 |


In [ ]:
from langchain_community.vectorstores import Chroma
import tiktoken

# ========== 토큰 제한 처리 ==========
# 임베딩 모델(text-embedding-3-small)은 최대 8,191 토큰까지 처리 가능하다.
# PDF 전체 페이지가 이 한도를 초과할 수 있으므로 미리 잘라 준다.
MAX_TOKENS = 8000
enc = tiktoken.encoding_for_model('text-embedding-3-small')

truncated = 0
for doc in all_documents:
    tokens = enc.encode(doc.page_content)
    if len(tokens) > MAX_TOKENS:
        doc.page_content = enc.decode(tokens[:MAX_TOKENS])
        truncated += 1
if truncated:
    print(f'⚠️ {truncated}개 문서가 토큰 제한 초과로 잘림')

# ========== Vector Store 생성 (페이지 단위) ==========
vectorstore = Chroma.from_documents(
    documents=all_documents,
    embedding=embeddings,
    collection_name='yes24_page'
)
print(f'Vector Store 생성 완료 (저장: {vectorstore._collection.count()}개)')

# Retriever 생성: 상위 3개 유사 문서를 반환
retriever = vectorstore.as_retriever(search_kwargs={'k': 3})


In [ ]:
# ========== 의미 기반 검색 테스트 ==========
# 키워드 검색에서 실패했던 질문들로 테스트한다.
test_queries = [
    ('빠른 배송', '→ "총알배송" 문서를 찾을 수 있는가?'),
    ('포인트 유효기간', '→ "영원한 YES포인트" 문서를 찾을 수 있는가?'),
    ('배송이 늦으면 보상', '→ "배송지연 보상제도" 문서를 찾을 수 있는가?'),
]

for query, desc in test_queries:
    results = retriever.invoke(query)
    source = os.path.basename(results[0].metadata.get('source', '')) if results else '없음'
    print(f'  "{query}" {desc}')
    print(f'    → 검색된 문서: {source}')
    print(f'    → 내용 (앞 80자): {results[0].page_content[:80]}...')
    print()

print('✅ 키워드가 정확히 일치하지 않아도 의미적으로 관련된 문서를 찾아낸다!')


---

## 5. 청킹(Chunking) 전략

### 5-1. 왜 청킹이 필요한가?

![image E](https://i.ibb.co/jkrBdwRd/image-e.png)

지금까지 문서를 **페이지 단위**로 Vector Store에 저장했다.
하지만 한 페이지에 **여러 주제**가 섞여 있을 수 있다.

"배송"을 질문했는데 페이지 전체가 반환되면, 반품/회원 정보까지 LLM에게 전달된다.
이런 **노이즈**는 답변 품질을 떨어뜨린다.

**청킹**: 긴 문서를 적절한 크기의 **조각(chunk)**으로 나누는 기법

| 파라미터 | 설명 | 비유 |
|---------|------|------|
| `chunk_size` | 조각의 최대 크기 (글자 수) | 카드 한 장에 적을 수 있는 글자 수 |
| `chunk_overlap` | 조각 간 겹치는 부분 | 카드 사이에 겹쳐 놓는 영역 (문맥 유지) |

<br>

| chunk_size | 장점 | 단점 | 적합한 상황 |
|:---:|------|------|------|
| 작음 (100~200) | 세밀한 검색 | 문맥 부족 | FAQ, 짧은 Q&A |
| 중간 (300~500) | 균형 | - | 정책 문서, 일반 안내 |
| 큼 (1000+) | 풍부한 문맥 | 노이즈 포함 | 논문, 기술 문서 |


In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

# ========== 청킹 적용 + 새 Vector Store ==========
# Yes24 정책 문서는 조건부 설명이 많으므로 chunk_size=300이 적절하다.
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=300,
    chunk_overlap=50,
    length_function=len,
    separators=['\n\n', '\n', '.', ' ', '']  # 이 순서대로 분할 시도
)

chunked_documents = text_splitter.split_documents(all_documents)
print(f'원본: {len(all_documents)}개 → 청킹 후: {len(chunked_documents)}개 조각')

# 청킹된 Vector Store 생성
vectorstore_chunked = Chroma.from_documents(
    documents=chunked_documents,
    embedding=embeddings,
    collection_name='yes24_chunked'
)
retriever_chunked = vectorstore_chunked.as_retriever(search_kwargs={'k': 3})
print(f'청킹 Vector Store 생성 완료 ({vectorstore_chunked._collection.count()}개 벡터)')


In [ ]:
# ========== 페이지 vs 청킹 검색 비교 ==========
query = '배송이 늦으면 보상받을 수 있나요?'

results_page = retriever.invoke(query)
results_chunk = retriever_chunked.invoke(query)

print(f'질문: {query}\n')
print(f'--- 페이지 단위 검색 (첫 결과: {len(results_page[0].page_content)}자) ---')
print(f'{results_page[0].page_content[:200]}...')

print(f'\n--- 청킹 단위 검색 (첫 결과: {len(results_chunk[0].page_content)}자) ---')
print(f'{results_chunk[0].page_content[:200]}...')

print('\n→ 청킹된 결과가 더 집중적이고 관련성이 높다!')


---

## 6. RAG 파이프라인 구축 (LangGraph)

### 6-1. 지금까지의 여정

```
① LLM만 → 환각 발생 (챕터 1)
② RAG = "답변 전에 자료를 먼저 찾아 읽어라" (챕터 2)
③ 키워드 검색 → 동의어 인식 불가 (챕터 3)
④ 임베딩 + Vector Store → 의미 기반 검색 (챕터 4)
⑤ 청킹으로 검색 정밀도 향상 (챕터 5)
⑥ 이 모든 것을 하나의 파이프라인으로! (이번 챕터)
```

### 6-2. LangGraph란?

LangChain 팀에서 개발한 **상태 기반 워크플로우 프레임워크**이다.
AI 애플리케이션의 복잡한 작업 흐름을 **그래프(Graph)** 구조로 설계할 수 있게 해준다.

> **💡 왜 LangGraph가 필요한가?**
>
> 단순한 RAG는 "검색 → 생성" 2단계이므로 함수 2개만 호출하면 된다.
> 하지만 실무에서는 훨씬 복잡한 흐름이 필요하다:
> - **조건부 분기**: 검색 결과가 없으면 → 다른 DB에서 재검색
> - **반복**: 답변 품질이 낮으면 → 검색 키워드를 바꿔서 재시도
> - **병렬 처리**: 여러 DB에서 동시에 검색
> - **Human-in-the-loop**: 특정 단계에서 사람의 승인을 받은 후 진행
>
> LangGraph는 이런 복잡한 워크플로우를 **시각적이고 체계적으로** 관리할 수 있게 해준다.


### 6-3. LangGraph의 핵심 구성요소

LangGraph는 세 가지 핵심 요소로 구성된다.

![image B](https://i.ibb.co/84DCh1dK/image-b.png)

#### ① State (상태)

그래프 전체에서 **공유되는 데이터 저장소**이다. 모든 노드가 이 State를 읽고 업데이트한다.

```python
from typing import TypedDict

class MyState(TypedDict):
    question: str    # 사용자 질문
    answer: str      # 생성된 답변
```

- `TypedDict`를 사용하여 어떤 데이터가 오가는지 **타입을 명시**한다
- 비유: **칠판** — 모든 작업자가 같은 칠판을 보며 읽고 쓴다

#### ② Node (노드)

State를 입력받아 **처리한 뒤 업데이트된 State를 반환**하는 함수이다.

```python
def my_node(state: MyState) -> MyState:
    # state에서 데이터를 읽어 처리하고
    result = some_processing(state['question'])
    # 업데이트할 필드만 반환하면 된다 (전체를 반환할 필요 없음)
    return {'answer': result}
```

- 각 노드는 **하나의 역할**만 담당한다 (검색 노드, 생성 노드 등)
- 비유: **작업자** — 칠판에서 정보를 읽고, 자기 작업 결과를 칠판에 적는다

#### ③ Edge (엣지)

노드 간의 **실행 순서(연결)**를 정의한다.

```python
workflow.add_edge(START, 'node_a')    # 시작 → node_a
workflow.add_edge('node_a', 'node_b') # node_a → node_b
workflow.add_edge('node_b', END)      # node_b → 끝
```

- `START`와 `END`는 LangGraph가 제공하는 특수 노드 (시작점, 종료점)
- 비유: **컨베이어 벨트** — 작업 순서를 결정한다

### 6-4. LangGraph 기본 문법 — 4단계 코드 패턴

LangGraph로 워크플로우를 만드는 코드는 항상 **동일한 4단계 패턴**을 따른다.

```python
# 1단계: State 정의 (TypedDict)
class MyState(TypedDict):
    ...

# 2단계: Node 함수 정의
def node_a(state: MyState) -> MyState:
    ...

# 3단계: Graph 구성 (노드 추가 + 엣지 연결)
workflow = StateGraph(MyState)
workflow.add_node('node_a', node_a)
workflow.add_edge(START, 'node_a')
workflow.add_edge('node_a', END)

# 4단계: 컴파일 + 실행
graph = workflow.compile()
result = graph.invoke({'question': '...'})
```

아래에서 이 패턴을 간단한 예제로 먼저 익힌 뒤, RAG 파이프라인에 적용한다.


### 6-5. LangGraph 기초 실습 — Hello World

RAG 파이프라인을 만들기 전에, 가장 간단한 예제로 LangGraph의 동작 방식을 이해하자.

목표: 사용자의 이름을 받아 **인사말을 생성**하는 2단계 워크플로우

```
START → [format 노드] → [greet 노드] → END
         이름을 정리      인사말 생성
```


In [ ]:
from typing import TypedDict
from langgraph.graph import StateGraph, START, END

# ========== 1단계: State 정의 ==========
# 이 그래프에서 오가는 데이터의 구조를 정의한다.
class GreetState(TypedDict):
    name: str       # 사용자 이름 (입력)
    formatted: str  # 정리된 이름 (중간 결과)
    greeting: str   # 최종 인사말 (출력)

# ========== 2단계: Node 함수 정의 ==========
# 각 노드는 state를 받아서, 업데이트할 필드만 딕셔너리로 반환한다.

def format_name(state: GreetState) -> GreetState:
    """이름을 정리하는 노드: 공백 제거 + 첫 글자 대문자"""
    name = state['name'].strip()
    return {'formatted': name}  # 'formatted' 필드만 업데이트

def greet(state: GreetState) -> GreetState:
    """인사말을 생성하는 노드"""
    greeting = f'안녕하세요, {state["formatted"]}님! LangGraph에 오신 걸 환영합니다.'
    return {'greeting': greeting}  # 'greeting' 필드만 업데이트

# ========== 3단계: Graph 구성 ==========
workflow = StateGraph(GreetState)

# 노드 추가: add_node('노드이름', 함수)
workflow.add_node('format', format_name)
workflow.add_node('greet', greet)

# 엣지 연결: add_edge(출발, 도착)
workflow.add_edge(START, 'format')   # 시작 → format
workflow.add_edge('format', 'greet') # format → greet
workflow.add_edge('greet', END)      # greet → 끝

# ========== 4단계: 컴파일 + 실행 ==========
hello_graph = workflow.compile()

# invoke: 초기 State를 넣으면, 모든 노드를 순서대로 실행하고 최종 State를 반환
result = hello_graph.invoke({'name': '  김싸피  '})

print(f'입력: "{result["name"]}"')
print(f'정리: "{result["formatted"]}"')
print(f'결과: "{result["greeting"]}"')
print('\n→ State가 노드를 거치며 단계적으로 업데이트되는 것을 확인!')


### 6-6. [참고] 조건부 엣지 (Conditional Edge)

LangGraph의 강력한 기능 중 하나는 **조건에 따라 다른 노드로 분기**할 수 있다는 것이다.

```python
# 조건부 분기 예시
def route_decision(state: MyState) -> str:
    """state 내용에 따라 다음 노드를 결정하는 함수"""
    if state['context'] == '':
        return 'fallback'    # 검색 결과 없음 → fallback 노드로
    else:
        return 'generate'    # 검색 결과 있음 → generate 노드로

workflow.add_conditional_edges(
    'retrieve',              # 출발 노드
    route_decision,          # 분기 판단 함수
    {'generate': 'generate', 'fallback': 'fallback'}  # 반환값 → 노드 매핑
)
```

```
              ┌─ context 있음 → [generate] → END
START → [retrieve] ─┤
              └─ context 없음 → [fallback] → END
```

> **💡 이 실습에서는 기본 엣지만 사용하지만,**
> 실무에서는 조건부 엣지로 "검색 실패 시 재검색", "답변 품질 낮으면 재생성" 등의
> 고급 워크플로우를 구현할 수 있다.

### 6-7. LangGraph로 RAG 파이프라인 구현

이제 Hello World에서 익힌 4단계 패턴을 **RAG 파이프라인에 그대로 적용**한다.

<!-- 🖼️ 이미지 위치 B: RAG LangGraph 파이프라인 다이어그램 -->

```
START → [retrieve 노드] → [generate 노드] → END
         질문으로 검색       검색 결과로 답변 생성
```

| Hello World | RAG 파이프라인 |
|:---:|:---:|
| `GreetState` | `RAGState` |
| `format_name` 노드 | `retrieve` 노드 (Vector Store 검색) |
| `greet` 노드 | `generate` 노드 (LLM 답변 생성) |


In [ ]:
from typing import TypedDict
from langgraph.graph import StateGraph, START, END

# ========== 1. State 정의 ==========
# 그래프 전체에서 공유되는 데이터 구조 ("칠판")
# 각 노드는 이 State를 읽고 업데이트한다.
class RAGState(TypedDict):
    question: str   # 사용자 질문
    context: str    # 검색된 문서 내용
    answer: str     # 생성된 답변

# ========== 2. retrieve 노드 ==========
# 역할: 질문으로 Vector Store에서 관련 문서를 검색한다.
def retrieve(state: RAGState) -> RAGState:
    """Vector Store에서 관련 문서를 검색하는 노드"""
    docs = retriever_chunked.invoke(state['question'])
    context = '\n\n'.join([doc.page_content for doc in docs])
    return {'context': context}

# ========== 3. generate 노드 ==========
# 역할: 검색된 문서를 참고하여 LLM이 답변을 생성한다.
def generate(state: RAGState) -> RAGState:
    """검색된 문서를 기반으로 답변을 생성하는 노드"""
    rag_prompt = ChatPromptTemplate.from_messages([
        ('system', '''당신은 온라인 서점 Yes24의 고객 서비스 상담원입니다.
다음 검색된 자료를 참고하여 고객의 질문에 정확하고 친절하게 답변해주세요.

[검색된 자료]
{context}

답변 규칙:
1. 검색된 자료를 기반으로 답변하세요
2. 자료에 없는 내용은 추측하지 마세요
3. 존댓말을 사용하세요'''),
        ('human', '{question}')
    ])
    messages = rag_prompt.format_messages(
        context=state['context'], question=state['question']
    )
    response = llm.invoke(messages)
    return {'answer': response.content}

# ========== 4. StateGraph 구성 ==========
workflow = StateGraph(RAGState)
workflow.add_node('retrieve', retrieve)
workflow.add_node('generate', generate)
workflow.add_edge(START, 'retrieve')
workflow.add_edge('retrieve', 'generate')
workflow.add_edge('generate', END)

rag_graph = workflow.compile()
print('LangGraph RAG 파이프라인 구성 완료!')


In [ ]:
# ========== RAG 파이프라인 실행 ==========
test_questions = [
    'Yes24에서 총알배송이 뭔가요? 서울에서 당일배송 마감 시간은?',
    '배송이 늦으면 보상받을 수 있나요?',
    'YES포인트는 유효기간이 있나요?',
    '신규 회원 혜택은 무엇인가요?',
]

for q in test_questions:
    result = rag_graph.invoke({'question': q})
    print(f'질문: {q}')
    print(f'답변: {result["answer"][:250]}')
    print('=' * 60)


In [ ]:
# ========== 최종 비교: LLM만 vs RAG ==========
comparison_q = 'Yes24 배송지연 보상제도는 어떻게 되나요? 보상 금액은?'

# LLM만
llm_only = llm.invoke([HumanMessage(content=comparison_q)])
# RAG
rag_result = rag_graph.invoke({'question': comparison_q})

print('=' * 60)
print('최종 비교: LLM만 vs RAG')
print('=' * 60)
print(f'\n질문: {comparison_q}')
print(f'\n--- LLM만 (환각 위험) ---')
print(llm_only.content[:300])
print(f'\n--- RAG (자료 기반) ---')
print(rag_result['answer'][:300])
print(f'\n정답: 주문 건당 YES포인트 2,000원 (실제 PDF 기준)')


---

## 정리

### 오늘 배운 전체 흐름

```
① LLM만 사용 → 환각 발생 (챕터 1)
② RAG = "답변 전에 자료를 먼저 찾아 읽어라" (챕터 2)
③ 키워드 검색 → 동의어 인식 불가 (챕터 3)
④ 임베딩 + Vector Store → 의미 기반 검색 (챕터 4)
⑤ 청킹으로 검색 정밀도 향상 (챕터 5)
⑥ LangGraph로 전체 파이프라인 자동화 (챕터 6)
```

### 핵심 개념 요약

| 개념 | 한 줄 정리 |
|------|----------|
| **RAG** | "답변 전에 자료를 먼저 찾아 읽어라" — 오픈북 시험 |
| **임베딩** | 텍스트를 숫자 벡터로 변환 → 의미가 비슷하면 벡터도 비슷 |
| **Vector Store** | 임베딩 벡터를 저장하고 유사도 검색하는 DB |
| **청킹** | 긴 문서를 적절한 크기로 잘라 검색 정밀도 향상 |
| **Retriever** | Vector Store에서 질문과 유사한 문서를 찾아 반환 |
| **LangGraph** | State + Node + Edge로 워크플로우를 구성하는 프레임워크 |

### RAG 파이프라인 전체 구성도

```
[준비 단계]
PDF → PyMuPDFLoader → 청킹(300자) → 임베딩 → ChromaDB 저장

[실행 단계 — LangGraph]
질문 → retrieve 노드(검색) → generate 노드(생성) → 정확한 답변
```

### 실습 안내

이제 `실습_4-1_RAG_기반_Customer_Service_AI_에이전트_개발.ipynb`를 열고,
오늘 배운 개념을 TODO 코드로 직접 구현해 보자.

---

### **Content License Agreement**

<font color='red'><b>**WARNING**</b></font> : 본 자료는 삼성청년SW·AI아카데미의 컨텐츠 자산으로, 보안서약서에 의거하여 어떠한 사유로도 임의로 복사, 촬영, 녹음, 복제, 보관, 전송하거나 허가 받지 않은 저장매체를 이용한 보관, 제3자에게 누설, 공개 또는 사용하는 등의 무단 사용 및 불법 배포 시 법적 조치를 받을 수 있습니다.
